<a href="https://colab.research.google.com/github/duruamobi/AAI2026/blob/main/Customer_Service_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CUSTOMER SERVICE CHATBOT

In [ ]:

# ============================================================
# CELL 1 — Install packages
# Run this first, then Runtime → Restart session, then run all
# ============================================================
!pip install -q langchain langchain-core langgraph langchain-google-genai


# ============================================================
# CELL 2 — API Setup
# In Colab: click the 🔑 key icon → Add new secret
# Name: Gemini_API_Key
# Value: your Gemini API key
# ============================================================

import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

api_key = userdata.get("Gemini_API_Key")
print("Secret found:", bool(api_key))

os.environ["GOOGLE_API_KEY"] = api_key

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    google_api_key=api_key,
)

print("✅ Gemini LLM ready!")


# ============================================================
# CELL 3 — Imports
# ============================================================

from typing import TypedDict, Literal
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

print("✅ Imports loaded!")


# ============================================================
# CELL 4 — System Prompts
# ============================================================

MAIN_SYSTEM_PROMPT = """
You are ShopMate, a friendly, professional, and context-aware customer service
and sales assistant for an online store.

Your responsibilities:
1. Help users with order-related questions
2. Help users with refunds, returns, and exchanges
3. Help users with product suggestions and recommendations

Personality and tone:
- Be warm, patient, and helpful
- Be concise but clear
- Sound professional, not robotic
- Show empathy when a customer has a problem
- Be confident when giving product suggestions
- Maintain context across the conversation and remember details the user already shared

Behavior rules:
- If the user asks about an order, help with order status, shipping updates, tracking, and delivery questions
- If the user asks about refunds or returns, explain policy, eligibility, timeline, and next steps
- If the user asks for product suggestions, ask 1-2 clarifying questions when needed, then recommend suitable products with short reasons
- If the request is unclear, ask a brief follow-up question before answering
- If the user provides details like order number, item type, budget, or preferences, REMEMBER and USE them in later turns — never ask again for something already provided
- Never invent specific order details like tracking numbers or delivery dates that were not provided
- If exact store policy is unknown, say it depends on store policy and give a reasonable general response

Goal:
Provide accurate, customer-friendly support while keeping the conversation natural and consistent.
""".strip()


REFUND_AGENT_PROMPT = """
You are the Refund Support Agent.

Your job is to help customers with refunds, returns, exchanges, and damaged-item issues.

Store Refund Policy:
- Items can be returned within 30 days of purchase in original condition
- Damaged or incorrect items qualify for a full refund or free replacement
- Refunds are processed within 5-7 business days after the return is received
- Digital products are non-refundable
- Sale items can only be exchanged unless damaged
- Customer needs their order number and reason to start a return

Guidelines:
- Always acknowledge the customer's frustration first before explaining policy
- Explain refund or return eligibility clearly using the policy above
- Ask for the order number only if the customer has not already provided it
- If the customer reports a damaged or incorrect item, apologize sincerely and explain next steps
- Reference any order number the customer already shared — do not ask again
- Keep answers short, clear, and supportive
""".strip()


ORDER_AGENT_PROMPT = """
You are the Order Support Agent.

Your job is to help customers with order status, shipping progress, delivery estimates,
tracking, and shipping-related questions.

Store Shipping Info:
- Standard shipping: 5-7 business days
- Express shipping: 2-3 business days
- Orders are processed within 1-2 business days after being placed
- Tracking becomes available once the order ships

Guidelines:
- Ask for an order number ONLY if the customer has not already provided one
- If the customer already gave an order number earlier, reference it naturally — do NOT ask again
- If there is a delay, acknowledge the inconvenience before explaining next steps
- Give realistic general delivery estimates when exact data is unavailable
- Remember everything shared earlier in the conversation
""".strip()


SUGGESTION_AGENT_PROMPT = """
You are the Product Suggestion Agent.

Your job is to recommend products based on the customer's needs, budget, preferences,
and intended use.

Product categories available:
Electronics, Home & Kitchen, Beauty & Skincare, Sports & Outdoors, Books & Stationery,
Toys & Games, Fashion & Accessories, Pet Supplies, Fitness Equipment, Office Supplies

Guidelines:
- If the request is broad, ask 1-2 short clarifying questions before recommending
- Once you have enough info, recommend 2-4 products or product categories
- Give a short reason for each recommendation (1 sentence)
- Use any preferences already shared (budget, age, occasion) — do NOT re-ask
- Format suggestions as a numbered list for easy reading
- Be helpful and enthusiastic but not pushy
""".strip()

print("✅ System prompts loaded!")


# ============================================================
# CELL 5 — State, Router & Agents
# ============================================================

class ChatState(TypedDict):
    messages: list
    intent: Literal["refund", "order", "suggestion", "general"]


def detect_intent(user_text: str) -> str:
    """
    Decision-making: detect what type of question the user asked
    and route to the correct agent.
    Priority: refund → order → suggestion → general
    """
    text = user_text.lower()

    refund_keywords = [
        "refund", "return", "exchange", "damaged", "broken", "defective",
        "wrong item", "incorrect item", "cancel order", "money back",
        "policy", "reimburse", "replacement"
    ]
    order_keywords = [
        "order", "shipping", "tracking", "delivery", "package",
        "arrive", "shipment", "shipped", "status", "late", "delayed",
        "where is my", "when will", "hasn't arrived", "not received"
    ]
    suggestion_keywords = [
        "recommend", "suggest", "suggestion", "best", "buy",
        "looking for", "need a product", "gift", "which one should i buy",
        "what should i get", "help me find", "shopping for", "ideas",
        "what do you recommend", "options", "compare", "good for"
    ]

    if any(k in text for k in refund_keywords):
        return "refund"
    if any(k in text for k in order_keywords):
        return "order"
    if any(k in text for k in suggestion_keywords):
        return "suggestion"
    return "general"


def router_node(state: ChatState) -> ChatState:
    last_user_message = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            last_user_message = msg.content
            break
    intent = detect_intent(last_user_message)
    # Show which agent is handling the question
    agent_labels = {
        "order":      "📦 Order Support Agent",
        "refund":     "💸 Refund Support Agent",
        "suggestion": "🎁 Product Suggestion Agent",
        "general":    "💬 General Support Agent",
    }
    print(f"   [{agent_labels[intent]}]")
    return {"messages": state["messages"], "intent": intent}


def order_agent(state: ChatState) -> ChatState:
    messages = state["messages"]
    system_msg = SystemMessage(content=MAIN_SYSTEM_PROMPT + "\n\n" + ORDER_AGENT_PROMPT)
    response = llm.invoke([system_msg] + messages)
    return {"messages": messages + [response], "intent": "order"}


def refund_agent(state: ChatState) -> ChatState:
    messages = state["messages"]
    system_msg = SystemMessage(content=MAIN_SYSTEM_PROMPT + "\n\n" + REFUND_AGENT_PROMPT)
    response = llm.invoke([system_msg] + messages)
    return {"messages": messages + [response], "intent": "refund"}


def suggestion_agent(state: ChatState) -> ChatState:
    messages = state["messages"]
    system_msg = SystemMessage(content=MAIN_SYSTEM_PROMPT + "\n\n" + SUGGESTION_AGENT_PROMPT)
    response = llm.invoke([system_msg] + messages)
    return {"messages": messages + [response], "intent": "suggestion"}


def general_agent(state: ChatState) -> ChatState:
    messages = state["messages"]
    system_msg = SystemMessage(
        content=MAIN_SYSTEM_PROMPT +
        "\n\nThe customer's request is unclear. Ask one short friendly "
        "clarifying question to understand how you can help them."
    )
    response = llm.invoke([system_msg] + messages)
    return {"messages": messages + [response], "intent": "general"}


def route_by_intent(state: ChatState) -> str:
    routes = {
        "order":      "order_agent",
        "refund":     "refund_agent",
        "suggestion": "suggestion_agent",
    }
    return routes.get(state["intent"], "general_agent")

print("✅ Agents and router defined!")


# ============================================================
# CELL 6 — Build Graph + MemorySaver
# ============================================================

graph_builder = StateGraph(ChatState)

graph_builder.add_node("router",           router_node)
graph_builder.add_node("order_agent",      order_agent)
graph_builder.add_node("refund_agent",     refund_agent)
graph_builder.add_node("suggestion_agent", suggestion_agent)
graph_builder.add_node("general_agent",    general_agent)

graph_builder.add_edge(START, "router")

graph_builder.add_conditional_edges(
    "router",
    route_by_intent,
    {
        "order_agent":      "order_agent",
        "refund_agent":     "refund_agent",
        "suggestion_agent": "suggestion_agent",
        "general_agent":    "general_agent",
    }
)

graph_builder.add_edge("order_agent",      END)
graph_builder.add_edge("refund_agent",     END)
graph_builder.add_edge("suggestion_agent", END)
graph_builder.add_edge("general_agent",    END)

# MemorySaver keeps the full conversation history across turns
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("✅ Graph compiled with MemorySaver!")


# ============================================================
# CELL 7 — Chat Helper
# ============================================================

def chat_with_bot(user_input: str, thread_id: str = "customer-1") -> str:
    """
    Sends the user's message to ShopMate and returns the response.
    thread_id keeps memory across the whole conversation.
    """
    config = {"configurable": {"thread_id": thread_id}}
    state  = {"messages": [HumanMessage(content=user_input)], "intent": "general"}
    result = graph.invoke(state, config=config)
    return result["messages"][-1].content

print("✅ chat_with_bot() ready!")


# ============================================================
# CELL 8 — START THE CHATBOT
# Run this cell to start chatting. Type your own questions!
# ============================================================

def run_chatbot():
    print("\n" + "=" * 60)
    print("🛍️  ShopMate — Customer Support Chatbot")
    print("=" * 60)
    print("Hello! I'm ShopMate, your customer service assistant.")
    print("I can help you with:\n")
    print("  📦  Order status & shipping questions")
    print("       e.g. 'Where is my order #12345?'")
    print("  💸  Refunds, returns & exchanges")
    print("       e.g. 'I want to return a damaged item'")
    print("  🎁  Product suggestions & recommendations")
    print("       e.g. 'Can you suggest a gift under $50?'")
    print("\nType 'quit' to end the chat.")
    print("=" * 60 + "\n")

    # Each session gets a unique thread so memory is fresh
    thread_id = "user-session-1"

    while True:
        # Get input from the user
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nShopMate: Thanks for visiting! Have a great day! 👋")
            break

        # Skip empty input
        if not user_input:
            continue

        # Exit commands
        if user_input.lower() in ("quit", "exit", "bye", "goodbye"):
            print("ShopMate: Thank you for reaching out! Have a wonderful day! 👋")
            break

        # Get and print the bot's response
        response = chat_with_bot(user_input, thread_id=thread_id)
        print(f"\nShopMate: {response}\n")


# Start the chatbot — type your questions below when prompted
run_chatbot()

Secret found: True
✅ Gemini LLM ready!
✅ Imports loaded!
✅ System prompts loaded!
✅ Agents and router defined!
✅ Graph compiled with MemorySaver!
✅ chat_with_bot() ready!

🛍️  ShopMate — Customer Support Chatbot
Hello! I'm ShopMate, your customer service assistant.
I can help you with:

  📦  Order status & shipping questions
       e.g. 'Where is my order #12345?'
  💸  Refunds, returns & exchanges
       e.g. 'I want to return a damaged item'
  🎁  Product suggestions & recommendations
       e.g. 'Can you suggest a gift under $50?'

Type 'quit' to end the chat.

